# 02 — Dependency Graph + Filter Panel

Builds a `networkx.DiGraph` from Terraform state or plan, then renders it as an interactive `ipycytoscape` widget with an `ipywidgets` filter panel.

**Requirements:** `pip install -e '.[notebook]'`

In [ ]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

import terra

FIXTURES = Path("../tests/fixtures")

## 1. Graph from State

In [ ]:
state = terra.load.state_local(FIXTURES / "state.json")
g_state = terra.graph.from_state(state)

print(f"Nodes: {g_state.number_of_nodes()}")
print(f"Edges: {g_state.number_of_edges()}")
print()
print("Nodes:")
for n, attrs in g_state.nodes(data=True):
    print(f"  {n}  (module={attrs['module']!r}, type={attrs['type']!r})")
print()
print("Edges (from → depends on):")
for src, dst in g_state.edges():
    print(f"  {src} → {dst}")

In [ ]:
# Render as interactive Cytoscape widget
terra.graph.render(g_state)

## 2. Graph from Plan

In [ ]:
plan = terra.load.plan_json(FIXTURES / "plan.json")
g_plan = terra.graph.from_plan(plan)

print(f"Nodes: {g_plan.number_of_nodes()}")
for n, attrs in g_plan.nodes(data=True):
    print(f"  {n}  actions={attrs['actions']}")

In [ ]:
terra.graph.render(g_plan)

## 3. Filter Panel

Dropdowns for **module**, **provider**, and **action** filter the nodes shown in the graph and the changes DataFrame simultaneously.

In [ ]:
import networkx as nx

changes = terra.frame.changes_df(plan)

# Collect unique filter values
modules   = ["(all)"] + sorted(set(changes["module_address"].fillna("")))
providers = ["(all)"] + sorted(changes["provider"].unique())
actions   = ["(all)"] + sorted({a for acts in changes["actions"] for a in acts})

dd_module   = widgets.Dropdown(options=modules,   description="Module:")
dd_provider = widgets.Dropdown(options=providers, description="Provider:")
dd_action   = widgets.Dropdown(options=actions,   description="Action:")

out_graph = widgets.Output()
out_table = widgets.Output()


def _filter(*_args):
    mask = changes["actions"].apply(lambda x: True)
    if dd_module.value != "(all)":
        mask &= changes["module_address"].fillna("") == dd_module.value
    if dd_provider.value != "(all)":
        mask &= changes["provider"] == dd_provider.value
    if dd_action.value != "(all)":
        mask &= changes["actions"].apply(lambda acts: dd_action.value in acts)

    filtered = changes[mask]
    keep_nodes = set(filtered["address"])
    sub = g_plan.subgraph(keep_nodes)

    with out_graph:
        out_graph.clear_output(wait=True)
        if sub.number_of_nodes() > 0:
            display(terra.graph.render(sub))
        else:
            print("(no resources match the current filter)")

    with out_table:
        out_table.clear_output(wait=True)
        display(filtered[["address", "type", "actions", "attr_diff"]])


dd_module.observe(_filter, names="value")
dd_provider.observe(_filter, names="value")
dd_action.observe(_filter, names="value")

_filter()  # initial render

display(
    widgets.VBox([
        widgets.HBox([dd_module, dd_provider, dd_action]),
        out_graph,
        out_table,
    ])
)

## 4. IPython Magic

Load the `terra` extension to enable `%%terraform` cell magic.

In [ ]:
%load_ext terra

In [ ]:
# Example — uncomment if terraform is on PATH and you have a working config:
# %%terraform plan -out=tfplan
# After success, the parsed Plan is bound to _plan automatically.
# terra.frame.summary(_plan)